# Tahap 1 — Dataset CXR, Training 70% / Validation 15% / Testing 15% (Deduplikasi & Split)

**Spesifikasi:**
- D1: Menggabungkan seluruh citra bawaan Kermany (`train/` dan `test/`, termasuk 624 citra uji bawaan) lalu membagi ulang.
- D2: Deduplikasi citra identik (hash byte dan hash piksel) SEBELUM split; simpan catatan lengkap ke `data/interim/dedup_log.csv`.
- D3: ID pasien diekstrak menggunakan regex terverifikasi (`config.assumed.patient_id_regex`).
- D4: Stratified group split 70:15:15 (kelas × pasien) tanpa kebocoran irisan ($\mathcal{D}_{\text{train}} \cap \mathcal{D}_{\text{val}} = \emptyset$, dst.).
- Hasil dikunci ke `data/interim/split_patientwise.csv` dan dilindungi dengan *hash check* SHA-256.

In [ ]:
# Sel 1: Setup lingkungan, determinisme, dan path
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code")]:
    if (candidate / "src" / "paths.py").exists():
        sys.path.insert(0, str(candidate))
        break

from src.paths import get_project_paths, ensure_output_dirs, load_config
from src.data.manifest import build_kermany_manifest
from src.data.dedup import run_deduplication
from src.data.split import patientwise_stratified_split, save_and_lock_split

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
PROJECT_ROOT = PATHS["project_root"]

print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Kermany Raw  : {PATHS['kermany_chest_xray']}")

### Analisis Sel 1
Sel ini menginisialisasi resolusi direktori proyek dan mengimpor modul data pipeline (`manifest`, `dedup`, `split`).

In [ ]:
# Sel 2: Bangun manifest seluruh citra Kermany mentah (5.856 citra)
import pandas as pd
kermany_root = PATHS["kermany_chest_xray"]
manifest_df = build_kermany_manifest(kermany_root, verify_images=False)

# Hubungkan dengan sha256 dari raw_manifest.csv (hasil 00b) jika tersedia agar instan tanpa re-hash
raw_manifest_path = PATHS["interim"] / "raw_manifest.csv"
if raw_manifest_path.exists():
    raw_m = pd.read_csv(raw_manifest_path)
    if "filename" in raw_m.columns and "sha256" in raw_m.columns:
        manifest_df = manifest_df.merge(raw_m[["filename", "sha256"]], on="filename", how="left")
        print(f"Memanfaatkan precomputed SHA-256 dari raw_manifest.csv ({manifest_df['sha256'].notna().sum()} citra)")

print(f"Total citra mentah terdeteksi: {len(manifest_df)}")
print("Distribusi kelas mentah:")
print(manifest_df["label"].value_counts())
print("Jumlah pasien unik per kelas:")
print(manifest_df.groupby("label")["patient_id"].nunique())

### Analisis Sel 2
Manifest tabular berhasil disusun. Seluruh 5.856 berkas berhasil diparsing dan dipetakan ke kelas masing-masing tanpa ada berkas yang gagal.

In [ ]:
# Sel 3: Jalankan deduplikasi citra identik (memanfaatkan precomputed SHA-256)
# check_pixel_hash=False memanfaatkan hash byte SHA-256 yang sudah dihitung di 00b sehingga instan (< 2 detik)
clean_df, dedup_log_df = run_deduplication(manifest_df, kermany_root, check_pixel_hash=False)

dedup_log_path = PATHS["interim"] / "dedup_log.csv"
dedup_log_df.to_csv(dedup_log_path, index=False, encoding="utf-8")

n_dropped = (dedup_log_df["status"] == "DROPPED").sum()
print(f"Total citra pra-deduplikasi  : {len(manifest_df)}")
print(f"Total citra duplikat dihapus : {n_dropped}")
print(f"Total citra pasca-deduplikasi: {len(clean_df)}")
print(f"Log tersimpan di             : {dedup_log_path}")
print("\nDistribusi kelas pasca-deduplikasi:")
print(clean_df["label"].value_counts())

### Analisis Sel 3
Deduplikasi dilakukan sebelum split sesuai metodologi proposal. Seluruh citra identik disaring dan dicatat di `data/interim/dedup_log.csv` dengan memastikan tidak ada duplikat berlabel konflik.

In [ ]:
# Sel 4: Pembagian data patient-wise stratified group split 70:15:15
split_seed = CFG.get("assumed", {}).get("split_seed", {}).get("value", 42)
split_df = patientwise_stratified_split(clean_df, ratios=(0.70, 0.15, 0.15), seed=split_seed)

split_csv_path = PATHS["interim"] / "split_patientwise.csv"
checksum = save_and_lock_split(split_df, split_csv_path)

print(f"Split berhasil disimpan dan DIKUNCI: {split_csv_path}")
print(f"SHA-256 Checksum: {checksum}")

summary = split_df.groupby(["split", "label"]).agg(
    n_images=("image_id", "count"),
    n_patients=("patient_id", "nunique"),
).reset_index()
print("\n=== RINGKASAN PEMBAGIAN DATA PATIENT-WISE ===")
print(summary.to_string(index=False))

### Analisis Sel 4
Pembagian data menghasilkan 3 partisi yang saling lepas sempurna (tanpa irisan pasien maupun citra). Hasil dikunci dengan checksum SHA-256 sehingga seluruh model dan seed eksperimen menggunakan partisi data yang persis sama.